# DoTime suites — overview

This notebook describes the two benchmark suites used in the paper and checks,
against the released data, every property the probe relies on: what an episode
contains, what the stored targets mean, where the query sits in time, which
episodes have a nonzero effect, and how the Continuous episodes were selected.

Each section follows the same order: a short explanation, a check, and how to
read the result. The time-index conversions are imported from
`probe.interfaces`, so the checks run on the same functions as the experiments.

| Suite | Version | Role in the paper |
|---|---|---|
| `dot-Identifiability-v1` | 1.1.0 | eight identification structures, one-step interventions (Tables 1 and 4) |
| `dot-Continuous-v1` | 1.0.0 | interventions held over a window, evaluated in-window and post-window (Effect Profile) |

Section 7 loads the superseded Identifiability **1.0.0** only for the
historical record. It is not used in any analysis.

## 1. Setup

Run from the repository root or from `notebooks/`. The Identifiability suite is
downloaded into `data/` on first use.

In [ ]:
import json
import sys
from collections import Counter
from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from huggingface_hub import snapshot_download
from dotime.benchmarks import load_benchmark, _SUITE_REGISTRY, _parse_suite_dir

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from probe.interfaces import get_query_step, reconstruct_window

pd.set_option("display.width", 120)

### Loading the two suites

Identifiability is pinned to an exact Hugging Face commit rather than a version
string: a tag can be re-pointed, a commit cannot. Continuous is loaded through
the DoTime registry, since its 1.0.0 release is unchanged.

In [ ]:
IDENT_NAME = "dot-Identifiability-v1"
IDENT_VERSION = "1.1.0"
IDENT_COMMIT = "03e330504eeac298a0063c7c694a902a9fd0180d"

CONT_NAME = "dot-Continuous-v1"
CONT_VERSION = "1.0.0"

ident_dir = ROOT / "data" / f"{IDENT_NAME}-{IDENT_COMMIT[:8]}"
snapshot_download(
    repo_id="thummd/dot-Identifiability-v1",
    repo_type="dataset",
    revision=IDENT_COMMIT,
    local_dir=ident_dir,
)

ident = _parse_suite_dir(replace(_SUITE_REGISTRY[IDENT_NAME], version=IDENT_VERSION), ident_dir)
cont = load_benchmark(CONT_NAME, version=CONT_VERSION)

print(f"Identifiability {IDENT_VERSION}: {len(ident)} episodes")
print(f"Continuous      {CONT_VERSION}: {len(cont)} episodes")

> **Reading the result.** Identifiability has 10,800 episodes and Continuous
> 9,999. Any other count means a different artifact was fetched; check the
> commit hash before running anything downstream.

## 2. What an episode contains

Every episode is a **pair of trajectories** simulated from the same causal
system: `x_obs` is what happened, `x_int` is what would have happened under
`do(A = v)`. Both are `(T, n_vars)` arrays. The episode also stores where to
look:

| Field | Meaning |
|---|---|
| `intervention.targets[0]` | the intervened variable (the treatment) |
| `intervention.times[0]` | the step at which the intervention starts |
| `intervention.values` | the imposed do-value |
| `query_target` | the variable we are asked about (the outcome) |
| `query_time` | *normalised* time of the question, in [0, 1] |
| `y_true` | the interventional outcome at the query |

In [ ]:
ep = ident[0]

print("structure       :", ep.structure)
print("x_obs shape     :", tuple(ep.x_obs.shape))
print("treatment var   :", int(ep.intervention.targets[0]))
print("intervention at :", int(ep.intervention.times[0]))
print("do-value        :", float(np.asarray(ep.intervention.values).reshape(-1)[0]))
print("queried var     :", int(ep.query_target[0]))
print("query_time      :", float(np.asarray(ep.query_time).reshape(-1)[0]))
print("metadata keys   :", list(ep.metadata.keys()))

> **Reading the result.** `query_time` is a fraction of the trajectory, not a
> step index; section 4.3 converts it. The metadata keys identify the release:
> 1.1.0 adds `diverged` and `pair_mode` to `tier`, `y_causal_effect` and
> `y_oracle`.

## 3. One table per suite

We walk each suite once and keep one row per episode, so every later check is a
short pandas expression. Two helpers come from `probe.interfaces`:

- `get_query_step` turns `query_time` into a step index. The suites scale it
  differently, by `T` in Identifiability and by `T - 1` in Continuous, so a
  single formula would read the wrong step in one of them.
- `reconstruct_window` recovers the end of a Continuous intervention, which the
  release does not store (section 5.2).

In [ ]:
def to_numpy(x):
    # Episode fields are torch tensors; convert without assuming a device.
    return x.detach().cpu().numpy() if hasattr(x, "detach") else np.asarray(x)


def scalar(x):
    return float(to_numpy(x).reshape(-1)[0])


def episode_table(suite, suite_name, with_window=False):
    rows = []
    for i, ep in enumerate(suite):
        x_obs, x_int = to_numpy(ep.x_obs), to_numpy(ep.x_int)
        t_start = int(ep.intervention.times[0])
        q = get_query_step(ep, suite_name)
        treatment = int(ep.intervention.targets[0])
        outcome = int(ep.query_target[0])

        row = {
            "episode": i,
            "structure": ep.structure,
            "n_vars": x_obs.shape[1],
            "t_start": t_start,
            "query_step": q,
            "offset": q - t_start,
            "treatment": treatment,
            "outcome": outcome,
            "self_intervention": treatment == outcome,
            "zeroed": bool(np.all(x_obs == 0) or np.all(x_int == 0)),
            "pre_onset_identical": np.array_equal(x_obs[:t_start], x_int[:t_start]),
            "y_true": scalar(ep.y_true),
            "interventional_level": float(x_int[q, outcome]),
            "paired_effect": float(x_int[q, outcome] - x_obs[q, outcome]),
            "stored_effect": scalar(ep.metadata["y_causal_effect"]),
        }
        if with_window:
            start, end = reconstruct_window(ep)
            row["duration"] = end - start
            row["in_window"] = start <= q < end
        rows.append(row)
    return pd.DataFrame(rows)


ident_df = episode_table(ident, IDENT_NAME)
cont_df = episode_table(cont, CONT_NAME, with_window=True)

print("Identifiability table:", ident_df.shape)
print("Continuous table     :", cont_df.shape)

> **Reading the result.** One row per episode. Building the tables takes a
> minute or so; everything below reads from them.

# 4. Identifiability v1.1.0

## 4.1 Structures and balance

The suite covers eight identification structures, and each changes what a
correct answer requires: `back_door` needs adjustment on an observed
confounder, `front_door` routes the effect through a mediator while the
confounder is hidden, and `instrumental_variable` hides the confounder but
offers an instrument. `observed_confounder` and `unobserved_confounder` have a
confounder of treatment and outcome but no causal path between them, so their
effect is identified and equal to zero; they act as null controls (section 4.5).

In [ ]:
display(ident_df["structure"].value_counts().sort_index().rename("episodes").to_frame())

> **Reading the result.** 1,350 episodes per structure. A single number pooled
> over the suite therefore weights every structure equally, which is why the
> paper also reports results per structure (Table 4).

## 4.2 Treatment and outcome pairs

Within a structure, is the causal question always the same pair of variables?
If not, a per-structure average would mix different questions.

In [ ]:
display(
    ident_df.groupby(["structure", "treatment", "outcome"])
    .size()
    .rename("episodes")
    .reset_index()
)

> **Reading the result.** Each structure uses a single treatment-outcome pair,
> with the treatment always in column 0. A per-structure result is one causal
> question repeated across many sampled systems.

## 4.3 Where the query sits in time

If `query_time` is converted to the wrong step, every metric silently scores the
wrong point. In this suite the question is asked at the moment of intervention,
so the recovered query step must equal `t_start` exactly; an off-by-one formula
would fail this check.

In [ ]:
print(ident_df["offset"].value_counts())

assert (ident_df["offset"] == 0).all(), "query step does not match t_start"
print("\nPASS: query_step == t_start on every episode")

> **Reading the result.** Every offset is zero, which confirms the
> `rint(query_time * T)` convention used by `get_query_step` (Appendix B). It also
> fixes the forecast horizon: from the onset through the query, inclusive, is a
> **single step**.

## 4.4 What the stored targets mean

Two fields look like the answer and are not the same thing:

- `y_true`, the interventional **level**: the value the outcome takes under the intervention;
- `metadata["y_causal_effect"]`, the **effect**: `x_int[q, Y] - x_obs[q, Y]`.

The probe scores effects, so reading the wrong field would compare a difference
with a level.

In [ ]:
targets = pd.DataFrame({
    "y_true == interventional level": [
        np.isclose(ident_df["y_true"], ident_df["interventional_level"]).mean()],
    "stored effect == paired difference": [
        np.isclose(ident_df["stored_effect"], ident_df["paired_effect"]).mean()],
    "stored effect == interventional level": [
        np.isclose(ident_df["stored_effect"], ident_df["interventional_level"]).mean()],
}, index=["fraction of episodes"]).T

display(targets)

> **Reading the result.** In 1.1.0 `y_true` is the level and the stored effect
> is the paired difference, as the names suggest. In the superseded 1.0.0 the
> stored effect matched the level instead (section 7).

## 4.5 Null-effect episodes

Because the query sits at the intervention step, the true effect is exactly zero
whenever the intervention has not reached the outcome at that step. This happens
for two reasons:

- in `observed_confounder` and `unobserved_confounder` there is **no causal
  path** from treatment to outcome;
- in `mediator` the path runs through a **lagged** edge, so it has not reached
  the outcome yet.

These three structures are therefore null controls. The other five also contain
episodes with a zero effect at the query. On a null episode the zero-effect
anchor is exact by construction, so the paper reports the Identifiability
headline on the nonzero-effect episodes only (Table 1) and keeps the null
controls separate (Table 4).

In [ ]:
ident_df["null_effect"] = ident_df["stored_effect"] == 0

null_by_structure = ident_df.groupby("structure").agg(
    episodes=("null_effect", "size"), null=("null_effect", "sum"))
null_by_structure["null fraction"] = (null_by_structure["null"] / null_by_structure["episodes"]).round(3)
display(null_by_structure)

# An exact-zero test and a small tolerance select the same episodes.
assert ident_df["null_effect"].sum() == (ident_df["stored_effect"].abs() <= 1e-9).sum()

print("Null-effect episodes   :", int(ident_df["null_effect"].sum()))
print("Nonzero-effect episodes:", int((~ident_df["null_effect"]).sum()))

> **Reading the result.** The three null structures are entirely null, and the
> other five have 15–34% null episodes: 5,635 null and 5,165 nonzero-effect
> episodes in total. Exact zero and a 1e-9 tolerance give the same count, so the
> filter `true_effect != 0` in `aggregate_metrics.py` selects exactly these
> 5,165 episodes.

## 4.6 Paired histories before the intervention

`x_obs` and `x_int` share the same system and the same noise realisation, so
before the intervention starts they must be identical. This is what makes the
paired difference a per-episode counterfactual effect, and it lets P0 and P1
share a single context window.

In [ ]:
print("Pre-onset identical:", int(ident_df["pre_onset_identical"].sum()), "/", len(ident_df))

> **Reading the result.** All episodes agree exactly before the onset. This
> property failed in 1.0.0, whose two arms were independent noise draws.

## 4.7 Degenerate episodes

Some simulated systems blow up numerically, and a diverged arm is stored as an
all-zero trajectory with no information. Release 1.1.0 flags these with
`diverged`; we check both the flag and the trajectories.

In [ ]:
n_diverged = sum(bool(ep.metadata["diverged"]) for ep in ident)

print("flagged as diverged  :", n_diverged)
print("all-zero trajectories:", int(ident_df["zeroed"].sum()))

> **Reading the result.** Both counts are zero, so no Identifiability episode is
> dropped for divergence. These are different from the null-effect episodes of
> section 4.5, whose trajectories are valid.

## 4.8 Hidden variables

`front_door`, `instrumental_variable` and `unobserved_confounder` depend on a
variable being **unobservable**. The generator hides it by zeroing that column in
`x_obs`; the check lists the fully zero columns per structure.

In [ ]:
for structure in sorted(ident_df["structure"].unique()):
    idx = ident_df.index[ident_df["structure"] == structure]
    patterns = Counter(
        tuple(np.where(np.all(to_numpy(ident[int(i)].x_obs) == 0, axis=0))[0].tolist())
        for i in idx
    )
    print(f"{structure:24} {patterns.most_common(2)}")

> **Reading the result.** The three structures with a hidden variable show one
> consistently zeroed column, and the other five show none, so no hidden
> variable leaks into the model's context.

## 4.9 One episode, drawn

Dashed lines are the observational world, solid lines the interventional one,
and the vertical line marks the intervention.

In [ ]:
ep = ident[0]
x_obs, x_int = to_numpy(ep.x_obs), to_numpy(ep.x_int)
t_start = int(ep.intervention.times[0])

fig, ax = plt.subplots(figsize=(9, 4))
for j in range(x_obs.shape[1]):
    ax.plot(x_obs[:, j], linestyle="--", color=f"C{j}", alpha=0.7, label=f"var {j} (obs)")
    ax.plot(x_int[:, j], color=f"C{j}", label=f"var {j} (int)")
ax.axvline(t_start, color="black", linestyle=":", label="intervention")
ax.set(xlabel="time step", ylabel="value", title=f"{ep.structure} — episode 0")
ax.legend(ncol=2, fontsize=8)
plt.show()

> **Reading the result.** The two worlds overlap before the vertical line and
> separate after it; the gap at the query point is the effect the probe asks
> the model to predict.

# 5. Continuous v1.0.0

Here the intervention is **held over a window** instead of applied once. This
lets the probe follow the effect while the intervention is active (in-window)
and after it ends (post-window), which is the Effect Profile of the paper.

In [ ]:
display(cont_df["structure"].value_counts().sort_index().rename("episodes").to_frame())
print("Total episodes:", len(cont_df))

> **Reading the result.** Three structures, balanced: `back_door`,
> `front_door` and `instrumental_variable`.

## 5.1 Targets

The same check as section 4.4, repeated because identical field names have
carried different meanings across releases.

In [ ]:
print("y_true == interventional level     :",
      np.isclose(cont_df["y_true"], cont_df["interventional_level"]).mean())
print("stored effect == paired difference :",
      np.isclose(cont_df["stored_effect"], cont_df["paired_effect"]).mean())
print("pre-onset identical                :", cont_df["pre_onset_identical"].mean())

> **Reading the result.** Continuous stores the paired difference and its arms
> agree before the onset, so both suites share one effect definition.

## 5.2 Recovering the intervention window

The release stores when the intervention **starts** but not when it ends. Under
a hard intervention the treatment is pinned to the do-value while the window is
active, so `reconstruct_window` walks forward from `t_start` until the
interventional treatment departs from the do-value; that first step is the
exclusive end `t_end` (Appendix B).

In [ ]:
display(cont_df["duration"].describe().to_frame("duration (steps)"))

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(cont_df["duration"], bins=40)
ax.set(xlabel="intervention duration (steps)", ylabel="episodes")
plt.show()

> **Reading the result.** Durations span 20 to 60 steps, the range stated in
> the paper's limitations.

## 5.3 When is the question asked?

Two things vary independently: how long the intervention lasts and how long
after its start the question is asked.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(cont_df["duration"], cont_df["offset"], s=3, alpha=0.2)
ax.plot([0, cont_df["duration"].max()], [0, cont_df["duration"].max()],
        color="black", linestyle=":", label="query at window end")
ax.set(xlabel="intervention duration", ylabel="query offset from onset")
ax.legend()
plt.show()

print("offset range:", int(cont_df["offset"].min()), "to", int(cont_df["offset"].max()))

> **Reading the result.** Points below the dotted line are queried while the
> intervention is active, points above it after it ended. This is why the
> Effect Profile bins episodes by offset, from the onset in-window and from the
> window end post-window, rather than by duration.

## 5.4 Regimes and self-queries

Two properties split the suite:

- **in-window vs post-window**: was the intervention still active at the query?
- **self vs non-self**: is the queried variable the treated one?

A self-query does not test propagation: in-window its answer is the do-value
itself. The paper therefore excludes self-queries and evaluates in-window and
post-window as separate regimes.

In [ ]:
display(
    cont_df.groupby(["in_window", "self_intervention"])
    .size()
    .rename("episodes")
    .reset_index()
)

for regime, flag in [("in-window", True), ("post-window", False)]:
    eligible = cont_df[(cont_df["in_window"] == flag) & ~cont_df["self_intervention"]]
    counts = eligible["structure"].value_counts().sort_index().to_dict()
    print(f"{regime:11} non-self: {len(eligible):,} episodes | per structure: {counts}")

> **Reading the result.** Every structure has more than 1,000 eligible episodes
> in each regime, which allows a balanced sample of 1,000 per structure.

## 5.5 The frozen selections

The runners evaluate a fixed sample of 1,000 non-self episodes per structure for
each regime, drawn once with seed 42 and stored in `results/selections/`. The
check confirms that the stored IDs have exactly those properties.

In [ ]:
by_id = cont_df.set_index("episode")

for regime, flag in [("in_window", True), ("post_window", False)]:
    sel = json.loads((ROOT / "results" / "selections" / f"continuous_{regime}_sel42.json").read_text())
    picked = by_id.loc[sel["episode_ids"]]

    assert len(picked) == 3000 and picked.index.is_unique
    assert (picked["structure"].value_counts() == 1000).all()
    assert (~picked["self_intervention"]).all() and (picked["in_window"] == flag).all()

    horizon = picked["offset"] + 1
    print(f"{regime:12} 3,000 episodes, 1,000 per structure, all non-self | "
          f"horizon {int(horizon.min())} to {int(horizon.max())} steps")

> **Reading the result.** Both selections match the protocol. In-window
> horizons stay below 64 steps, which is why Toto-1.0, whose first 64-step patch
> is decoded before the treatment is clamped, cannot respond (Appendix C).

# 6. Side-by-side summary

The properties that decide how each suite has to be handled.

In [ ]:
summary = pd.DataFrame({
    f"Identifiability {IDENT_VERSION}": {
        "episodes": len(ident_df),
        "structures": ident_df["structure"].nunique(),
        "trajectory length": int(ident[0].length),
        "intervention": "single step",
        "query_time scale": "T",
        "query offset": "always 0",
        "forecast horizon": "1",
        "self-intervention episodes": int(ident_df["self_intervention"].sum()),
        "zeroed episodes": int(ident_df["zeroed"].sum()),
        "evaluated in the paper": "5,165 nonzero-effect episodes (headline)",
    },
    f"Continuous {CONT_VERSION}": {
        "episodes": len(cont_df),
        "structures": cont_df["structure"].nunique(),
        "trajectory length": int(cont[0].length),
        "intervention": "window of 20-60 steps",
        "query_time scale": "T - 1",
        "query offset": f"{int(cont_df['offset'].min())} to {int(cont_df['offset'].max())}",
        "forecast horizon": "variable",
        "self-intervention episodes": int(cont_df["self_intervention"].sum()),
        "zeroed episodes": int(cont_df["zeroed"].sum()),
        "evaluated in the paper": "3,000 per regime (1,000 per structure)",
    },
})
display(summary)

> **Reading the result.** The `query_time` scale and the forecast horizon differ
> between the suites, so any shared code has to branch on the suite name, as
> `get_query_step` does.

# 7. Historical record — the superseded Identifiability 1.0.0

**This is the only place the old release is loaded, and it is used in no
analysis.** It keeps the reasons for migrating to 1.1.0 reproducible.

The DoTime documentation lists, among others, these problems of the 1.0.0
release:

1. the two arms were **independent noise draws**, so `x_int - x_obs` was not a
   per-episode counterfactual effect;
2. `x_obs` was stored with its columns in a different order from `x_int`;
3. hidden variables were not masked in `x_obs`;
4. `metadata["y_causal_effect"]` stored the interventional level, not the effect.

In [ ]:
ident_old = load_benchmark(IDENT_NAME, version="1.0.0")
old_df = episode_table(ident_old, IDENT_NAME)

print("Old episodes:", len(old_df))
print("Old metadata keys:", sorted(ident_old[0].metadata.keys()))
print("New metadata keys:", sorted(ident[0].metadata.keys()))

> **Reading the result.** Same episode count; `diverged` and `pair_mode` exist
> only in 1.1.0, which is the quickest way to tell the releases apart.

## 7.1 The problems, measured

One comparison per problem on both releases. Zeroed episodes are excluded from
the hidden-variable check, since their columns are all zero anyway.

In [ ]:
HIDDEN_STRUCTURES = {"front_door", "instrumental_variable", "unobserved_confounder"}


def hidden_masked_fraction(suite, table):
    # Among non-zeroed hidden-variable episodes, how many have a fully zero column?
    idx = table.index[table["structure"].isin(HIDDEN_STRUCTURES) & ~table["zeroed"]]
    if len(idx) == 0:
        return float("nan")
    return sum(np.any(np.all(to_numpy(suite[int(i)].x_obs) == 0, axis=0)) for i in idx) / len(idx)


def release_properties(suite, table):
    return {
        "stored effect == paired difference":
            np.isclose(table["stored_effect"], table["paired_effect"]).mean(),
        "stored effect == interventional level":
            np.isclose(table["stored_effect"], table["interventional_level"]).mean(),
        "pre-onset histories identical": table["pre_onset_identical"].mean(),
        "hidden variable masked": hidden_masked_fraction(suite, table),
        "zeroed episodes": int(table["zeroed"].sum()),
    }


display(pd.DataFrame({
    "v1.0.0 (superseded)": release_properties(ident_old, old_df),
    "v1.1.0 (in use)": release_properties(ident, ident_df),
}))

> **Reading the result.** The first four rows are fractions of episodes. In
> 1.0.0 the stored effect matches the level, the histories differ before the
> onset and the hidden columns are visible; 504 episodes are zeroed. In 1.1.0
> every property holds and no episode is zeroed.

## 7.2 Consequences for this project

The 1.0.0 release is a frozen, citable artifact, so DoTime fixed it in a new
version rather than in place. For this project:

- all results are reported on **1.1.0**, pinned to the commit in section 1;
- numbers computed on 1.0.0 before the migration are not comparable and were
  discarded;
- published baselines computed on 1.0.0 share these issues, so they are not
  compared with the paired effects reported here.